In [ ]:
# OpenCode / OpenAI-compatible notebook setup
# In your notebook's Python environment: pip install openai python-dotenv pyyaml
# This kernel's working directory becomes the repository root.
import os as _nb_os
from pathlib import Path as _NotebookPath
from dotenv import load_dotenv as _notebook_load_dotenv

CHAPTER = "s14_mcp_plugin"
PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if (directory / CHAPTER / "code.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ so code.py can be located"
    )
_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)
__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")

# Change this to True and rerun the last code cell to start the interactive CLI.
# Run All is intentionally non-interactive.
RUN_AGENT_INTERACTIVELY = False

# The original .env naming also works; OPENCODE_* names are optional.
_notebook_key = _nb_os.getenv("OPENCODE_API_KEY") or _nb_os.getenv("ANTHROPIC_API_KEY")
_notebook_url = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or _nb_os.getenv("ANTHROPIC_BASE_URL")
    or "https://opencode.ai/inference/openai/v1"
).rstrip("/")
if _notebook_url.endswith("/chat/completions"):
    _notebook_url = _notebook_url[:-len("/chat/completions")]
_notebook_model = _nb_os.getenv("OPENCODE_MODEL") or _nb_os.getenv("MODEL_ID")
if not _notebook_key:
    raise RuntimeError("Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env")
if not _notebook_model:
    raise RuntimeError("Set MODEL_ID (or OPENCODE_MODEL) in the root .env")
_nb_os.environ["OPENCODE_API_KEY"] = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"] = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"] = _notebook_model
print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")


In [ ]:
# OpenCode adapter: expose the Anthropic-shaped lesson API on top of OpenAI Chat Completions.
# The original .py remains untouched. This module mapping also supports s15/s16,
# which load earlier chapter .py files from disk via importlib.
import json as _compat_json
import sys as _compat_sys
import types as _compat_types
from openai import OpenAI as _OpenAI


def _compat_field(block, key, default=None):
    """Read a content field from either a provider object or a Python dict."""

    return block.get(key, default) if isinstance(block, dict) else getattr(block, key, default)


def _compat_text(value):
    """Convert Anthropic-style text blocks into text for Chat Completions."""

    if isinstance(value, str):
        return value
    if value is None:
        return ""
    if isinstance(value, list):
        return "\n".join(
            _compat_text(_compat_field(item, "text", _compat_field(item, "content", "")))
            for item in value
            if _compat_field(item, "type") != "tool_use"
        )
    if isinstance(value, dict):
        return _compat_text(value.get("text", value.get("content", _compat_json.dumps(value))))
    return str(value)


def _compat_history(messages, system=None):
    """Translate assistant tool calls and user tool results without losing call IDs."""

    result = []
    if system:
        result.append({"role": "system", "content": _compat_text(system)})

    for message in messages:
        role = _compat_field(message, "role")
        content = _compat_field(message, "content", "")
        if role == "assistant":
            if not isinstance(content, list):
                result.append({"role": "assistant", "content": _compat_text(content)})
                continue
            text_blocks = []
            tool_calls = []
            for block in content:
                kind = _compat_field(block, "type")
                if kind == "tool_use":
                    arguments = _compat_field(block, "input", {})
                    tool_calls.append({
                        "id": _compat_field(block, "id"),
                        "type": "function",
                        "function": {
                            "name": _compat_field(block, "name"),
                            "arguments": _compat_json.dumps(arguments, ensure_ascii=False),
                        },
                    })
                elif kind == "text":
                    text_blocks.append(_compat_field(block, "text", ""))
            converted = {"role": "assistant", "content": "\n".join(text_blocks) or None}
            if tool_calls:
                converted["tool_calls"] = tool_calls
            if converted["content"] is None and not tool_calls:
                converted["content"] = ""
            result.append(converted)
        elif role == "user" and isinstance(content, list):
            user_text = []
            tool_results = []
            for block in content:
                if _compat_field(block, "type") == "tool_result":
                    tool_results.append({
                        "role": "tool",
                        "tool_call_id": _compat_field(block, "tool_use_id"),
                        "content": _compat_text(_compat_field(block, "content", "")),
                    })
                else:
                    user_text.append(_compat_text(block))
            result.extend(tool_results)
            if user_text:
                result.append({"role": "user", "content": "\n".join(user_text)})
        else:
            result.append({"role": role, "content": _compat_text(content)})
    return result


class _NotebookMessages:
    """Translate lessons' messages.create calls to OpenAI-compatible requests."""

    def __init__(self, client):
        self._client = client

    def create(self, *, model, messages, system=None, tools=None, max_tokens=None, **kwargs):
        """Return Anthropic-shaped text/tool blocks with original call identifiers."""

        request = {"model": model, "messages": _compat_history(messages, system)}
        if tools:
            request["tools"] = [
                {
                    "type": "function",
                    "function": {
                        "name": tool["name"],
                        "description": tool.get("description", ""),
                        "parameters": tool.get(
                            "input_schema", {"type": "object", "properties": {}}
                        ),
                    },
                }
                for tool in tools
            ]
        if max_tokens is not None:
            request["max_tokens"] = max_tokens

        response = self._client.chat.completions.create(**request)
        choice = response.choices[0]
        reply = choice.message
        blocks = []
        if reply.content:
            blocks.append(_compat_types.SimpleNamespace(type="text", text=reply.content))
        for call in reply.tool_calls or []:
            try:
                arguments = _compat_json.loads(call.function.arguments or "{}")
                if not isinstance(arguments, dict):
                    arguments = {}
            except (ValueError, TypeError):
                arguments = {}
            blocks.append(_compat_types.SimpleNamespace(
                type="tool_use", id=call.id, name=call.function.name, input=arguments
            ))
        usage = getattr(response, "usage", None)
        return _compat_types.SimpleNamespace(
            content=blocks,
            stop_reason=(
                "max_tokens" if choice.finish_reason == "length"
                else "tool_use" if reply.tool_calls
                else "end_turn"
            ),
            usage=_compat_types.SimpleNamespace(
                input_tokens=getattr(usage, "prompt_tokens", 0) or 0,
                output_tokens=getattr(usage, "completion_tokens", 0) or 0,
            ),
        )


class _NotebookAnthropic:
    """Provide the old lesson client interface over the configured OpenCode endpoint."""

    def __init__(self, base_url=None, **kwargs):
        url = _nb_os.getenv("OPENCODE_BASE_URL") or base_url or _notebook_url
        self._client = _OpenAI(api_key=_notebook_key, base_url=url)
        self.messages = _NotebookMessages(self._client)


_anthropic_compat = _compat_types.ModuleType("anthropic")
_anthropic_compat.Anthropic = _NotebookAnthropic
_compat_sys.modules["anthropic"] = _anthropic_compat


In [ ]:
#!/usr/bin/env python3
"""
s14: MCP Tools - discover external tools and add them to the agent loop.

Run:  python s14_mcp_plugin/code.py
Need: pip install anthropic python-dotenv + .env with ANTHROPIC_API_KEY

    connect_mcp("docs")
              |
              v
    +------------------+     tools/list     +------------------+
    | Agent Harness    | <----------------- | MCP server       |
    |                  |                    | docs             |
    | built-in tools   |     tools/call     |                  |
    | + MCP tools      | -----------------> | search           |
    +--------+---------+                    | get_version      |
             |                              +------------------+
             v
    +-----------------------------------------------+
    | bash | read | write | edit | glob | connect  |
    | mcp__docs__search | mcp__docs__get_version   |
    +-----------------------------------------------+
"""


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import glob
import os
import re
import subprocess
from pathlib import Path

try:

    import readline
    readline.parse_and_bind("set bind-tty-special-chars off")
#
except ImportError:

    pass
#

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if (os.getenv("ANTHROPIC_BASE_URL")):

    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
#

WORKDIR = Path.cwd()
client  = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL   = os.environ["MODEL_ID"]

ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax, and prefer dedicated file tools for file operations")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)
BASE_SYSTEM        = (
    f"You are a coding agent at {WORKDIR}. Environment: {ENVIRONMENT_PROMPT}. "
    "Use built-in and connected MCP "
    "tools to solve tasks. Call connect_mcp before using a server."
)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### From s04: base tools
#

def run_bash(command:str) -> str:

    try:

        result = subprocess.run(
            command,
            shell          = True,
            cwd            = WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        output = (result.stdout + result.stderr).strip()
        output = (output[:50000]) if (output) else ("(no output)")
        if (result.returncode):

            return (f"Error: command exited with status {result.returncode}\n{output}")
        #
        return (output)
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
    except OSError as exc:

        return (f"Error: {type(exc).__name__}: {exc}")
    #
#


def run_read(path:str, limit:int|None=None) -> str:

    try:

        lines = (WORKDIR / path).resolve().read_text(encoding="utf-8").splitlines()
        if (limit  and  limit < len(lines)):

            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        #
        return ("\n".join(lines))
    #
    except Exception as exc:

        return (f"Error: {exc}")
    #
#


def run_write(path:str, content:str) -> str:

    try:

        target = (WORKDIR / path).resolve()
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return (f"Wrote {len(content)} bytes to {path}")
    #
    except Exception as exc:

        return (f"Error: {exc}")
    #
#


def run_edit(path:str, old_text:str, new_text:str) -> str:

    try:

        target  = (WORKDIR / path).resolve()
        content = target.read_text(encoding="utf-8")
        count   = content.count(old_text)
        if (count != 1):

            return (f"Error: Expected 1 occurrence, found {count}")
        #
        target.write_text(content.replace(old_text, new_text), encoding="utf-8")
        return (f"Edited {path}")
    #
    except Exception as exc:

        return (f"Error: {exc}")
    #
#


def run_glob(pattern:str) -> str:

    try:

        matches = sorted({
            match
            for match in glob.glob(pattern, root_dir=WORKDIR, recursive=True)
            if (WORKDIR / match).resolve().is_relative_to(WORKDIR.resolve())
        })
        shown   = matches[:200]
        if (len(matches) > 200):

            shown.append("... (more matches omitted; narrow the pattern)")
        #
        return (("\n".join(shown)) if (shown) else ("(no matches)"))
    #
    except Exception as exc:

        return (f"Error: {exc}")
    #
#


BASE_TOOLS = [
    {
        "name"         : "bash",
        "description"  : "Run a shell command.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"command": {"type": "string"}},
            "required"   : ["command"],
        },
    },
    {
        "name"         : "read_file",
        "description"  : "Read file contents.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"  : {"type": "string"},
                "limit" : {"type": "integer"},
            },
            "required"   : ["path"],
        },
    },
    {
        "name"         : "write_file",
        "description"  : "Write content to a file.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"    : {"type": "string"},
                "content" : {"type": "string"},
            },
            "required"   : ["path", "content"],
        },
    },
    {
        "name"         : "edit_file",
        "description"  : "Replace exact text once.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"     : {"type": "string"},
                "old_text" : {"type": "string"},
                "new_text" : {"type": "string"},
            },
            "required"   : ["path", "old_text", "new_text"],
        },
    },
    {
        "name"         : "glob",
        "description"  : "Find files by glob pattern; ** matches recursively.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"pattern": {"type": "string"}},
            "required"   : ["pattern"],
        },
    },
]

BASE_HANDLERS = {
    "bash"       : run_bash,
    "read_file"  : run_read,
    "write_file" : run_write,
    "edit_file"  : run_edit,
    "glob"       : run_glob,
}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### New in s14: MCP discovery and dispatch
#

class MCPClient:

    """Small in-process stand-in for MCP tools/list and tools/call."""

    def __init__(self, name:str):

        self.name = name
        self.tools: list[dict] = []
        self._handlers: dict[str, callable] = {}
    #

    def register(self, tool_defs:list[dict], handlers:dict[str, callable]):

        names = [tool.get("name") for tool in tool_defs]
        if (any(not isinstance(name, str)  or  not name for name in names)):

            raise (ValueError("Every MCP tool needs a non-empty name"))
        #
        if (len(set(names)) != len(names)):

            raise (ValueError(f"Duplicate MCP tool name on server {self.name!r}"))
        #
        missing = [name for name in names if name not in handlers]
        if (missing):

            raise (ValueError(f"Missing MCP handlers: {', '.join(missing)}"))
        #
        self.tools     = list(tool_defs)
        self._handlers = dict(handlers)
    #

    def call_tool(self, tool_name:str, args:dict) -> str:

        handler = self._handlers.get(tool_name)
        if (not handler):

            return (f"MCP error: unknown tool '{tool_name}'")
        #
        try:

            return (str(handler(**args)))
        #
        except Exception as exc:

            return (f"MCP error: {type(exc).__name__}: {exc}")
        #
    #
#


mcp_clients: dict[str, MCPClient] = {}
mcp_tool_policies: dict[str, str] = {}
_DISALLOWED_CHARS = re.compile(r"[^a-zA-Z0-9_-]")

# Authorization comes from host configuration, never server descriptions.
MCP_HOST_POLICY = {
    ("docs", "search")      : "allow",
    ("docs", "get_version") : "allow",
    ("deploy", "status")    : "allow",
    ("deploy", "trigger")   : "confirm",
}


def normalize_mcp_name(name:str) -> str:

    """Replace characters outside the model tool-name alphabet."""
    normalized = _DISALLOWED_CHARS.sub("_", name)
    if (not normalized):

        raise (ValueError("MCP names cannot normalize to an empty string"))
    #
    return (normalized)
#


def _mock_server_docs() -> MCPClient:

    server = MCPClient("docs")
    server.register(
        tool_defs = [
            {
                "name"        : "search",
                "description" : "Search the documentation.",
                "inputSchema" : {
                    "type"       : "object",
                    "properties" : {"query": {"type": "string"}},
                    "required"   : ["query"],
                },
                "annotations" : {"readOnlyHint": True},
            },
            {
                "name"        : "get_version",
                "description" : "Get the documentation API version.",
                "inputSchema" : {"type": "object", "properties": {}},
                "annotations" : {"readOnlyHint": True},
            },
        ],
        handlers  = {
            "search"      : lambda query: f"[docs] Found 3 results for '{query}'",
            "get_version" : lambda: "[docs] API v2.1.0",
        },
    )
    return (server)
#


def _mock_server_deploy() -> MCPClient:

    server = MCPClient("deploy")
    server.register(
        tool_defs = [
            {
                "name"        : "trigger",
                "description" : "Trigger a deployment.",
                "inputSchema" : {
                    "type"       : "object",
                    "properties" : {"service": {"type": "string"}},
                    "required"   : ["service"],
                },
                "annotations" : {"destructiveHint": True},
            },
            {
                "name"        : "status",
                "description" : "Check deployment status.",
                "inputSchema" : {
                    "type"       : "object",
                    "properties" : {"service": {"type": "string"}},
                    "required"   : ["service"],
                },
                "annotations" : {"readOnlyHint": True},
            },
        ],
        handlers  = {
            "trigger" : lambda service: f"[deploy] Triggered: {service}",
            "status"  : lambda service: f"[deploy] {service}: running (v1.4.2)",
        },
    )
    return (server)
#


MOCK_SERVERS = {
    "docs"   : _mock_server_docs,
    "deploy" : _mock_server_deploy,
}


def connect_mcp(name:str) -> str:

    if (name in mcp_clients):

        return (f"MCP server '{name}' already connected")
    #
    factory = MOCK_SERVERS.get(name)
    if (not factory):

        return (f"Unknown server '{name}'. Available: {', '.join(MOCK_SERVERS)}")
    #
    server            = factory()
    mcp_clients[name] = server
    names             = ", ".join(tool["name"] for tool in server.tools)
    print(f"  [mcp] connected: {name} -> {names}")
    return (
        f"Connected to MCP server '{name}'. "
        f"Discovered {len(server.tools)} tools: {names}"
    )
#


def run_connect_mcp(name:str) -> str:

    return (connect_mcp(name))
#


CONNECT_TOOL = {
    "name"         : "connect_mcp",
    "description"  : "Connect to an MCP server and discover its tools.",
    "input_schema" : {
        "type"       : "object",
        "properties" : {"name": {"type": "string", "enum": ["docs", "deploy"]}},
        "required"   : ["name"],
    },
}

BUILTIN_TOOLS    = [*BASE_TOOLS, CONNECT_TOOL]
BUILTIN_HANDLERS = {**BASE_HANDLERS, "connect_mcp": run_connect_mcp}




In [ ]:
def assemble_tool_pool() -> tuple[list[dict], dict[str, callable]]:

    """Combine built-in tools with every connected server tool."""
    global mcp_tool_policies
    tools = list(BUILTIN_TOOLS)
    handlers = dict(BUILTIN_HANDLERS)
    policies: dict[str, str] = {}
    origins = {
        tool["name"]: f"built-in tool {tool['name']!r}"
        for tool in tools
    }

    for server_name, server in mcp_clients.items():

        safe_server = normalize_mcp_name(server_name)
        for tool_def in server.tools:

            raw_name  = tool_def["name"]
            safe_tool = normalize_mcp_name(raw_name)
            prefixed  = f"mcp__{safe_server}__{safe_tool}"
            if (len(prefixed) > 64):

                raise (ValueError(f"MCP tool name is longer than 64 characters: {prefixed}"))
            #
            origin = f"MCP tool {server_name!r}/{raw_name!r}"
            if (prefixed in origins):

                raise (ValueError(
                    "MCP tool name collision after normalization: "
                    f"{prefixed!r} maps both {origins[prefixed]} and {origin}"
                ))
            #
            schema = tool_def.get("inputSchema", {})
            if (not isinstance(schema, dict)  or  schema.get("type", "object") != "object"):

                raise (ValueError(f"Invalid input schema for {origin}"))
            #
            origins[prefixed] = origin
            tools.append({
                "name"         : prefixed,
                "description"  : tool_def.get("description", ""),
                "input_schema" : schema,
            })
            handlers[prefixed] = (
                lambda *, client=server, tool=raw_name, **kwargs:
                client.call_tool(tool, kwargs)
            )
            policies[prefixed] = MCP_HOST_POLICY.get(
                (server_name, raw_name), "confirm"
            )
        #
    #

    mcp_tool_policies = policies
    return (tools, handlers)
#


def assemble_system_prompt() -> str:

    if (not mcp_clients):

        return (BASE_SYSTEM)
    #
    return (BASE_SYSTEM + "\n\nConnected MCP servers: " + ", ".join(mcp_clients))


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### From s04: hooks and permission checks
#

HOOKS                    = {"UserPromptSubmit": [], "PreToolUse": [], "PostToolUse": [], "Stop": []}
DENY_LIST                = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if="]
DESTRUCTIVE_COMMAND_WORD = re.compile(
    r"(?i)(?:^|[;&|()\n])\s*(?:rm|del)(?=\s|$|[;&|()])"
)
DESTRUCTIVE              = ["rm ", "> /etc/", "chmod 777"]


def contains_destructive_command(command:str) -> bool:

    return (bool(DESTRUCTIVE_COMMAND_WORD.search(command)))
#


def register_hook(event:str, callback):

    HOOKS[event].append(callback)
#


def trigger_hooks(event:str, *args):

    for callback in HOOKS[event]:

        result = callback(*args)
        if (result is not None):

            return (result)
        #
    #
    return (None)
#


def permission_hook(block):

    if (block.name == "bash"):

        command = block.input.get("command", "")
        for pattern in DENY_LIST:

            if (pattern in command):

                return (f"Permission denied by deny list: {pattern}")
            #
        #
        if (contains_destructive_command(command)  or  any(
            keyword in command for keyword in DESTRUCTIVE
        )):

            print(f"\n[permission] {block.name}({block.input})")
            if (input("Allow? [y/N] ").strip().lower() not in {"y", "yes"}):

                return ("Permission denied by user")
            #
        #
    #

    if (block.name in {"read_file", "write_file", "edit_file"}):

        raw_path = block.input.get("path", "")
        if (not (WORKDIR / raw_path).resolve().is_relative_to(WORKDIR.resolve())):

            print(f"\n[permission] {block.name}({block.input})")
            if (input("Allow? [y/N] ").strip().lower() not in {"y", "yes"}):

                return ("Permission denied by user")
            #
        #
    #

    if (block.name.startswith("mcp__")):

        policy = mcp_tool_policies.get(block.name, "confirm")
        if (policy != "allow"):

            print(f"\n[permission] External tool {block.name}({block.input})")
            if (input("Allow? [y/N] ").strip().lower() not in {"y", "yes"}):

                return ("Permission denied by user")
            #
        #
    #
    return (None)
#


def log_hook(block):

    preview = str(list(block.input.values())[:2])[:60]
    print(f"[hook] {block.name}({preview})")
    return (None)
#


def large_output_hook(block, output):

    if (len(str(output)) > 100000):

        print(f"[hook] Large output from {block.name}: {len(str(output))} chars")
    #
    return (None)
#


def context_hook(query:str):

    print(f"[hook] UserPromptSubmit: working in {WORKDIR}")
    return (None)
#


def summary_hook(messages:list):

    tool_count = sum(
        1
        for message in messages
        for block in (
            (message.get("content"))
            if (isinstance(message.get("content"), list))
            else ([])
        )
        if isinstance(block, dict)  and  block.get("type") == "tool_result"
    )
    print(f"[hook] Stop: session used {tool_count} tool calls")
    return (None)
#


register_hook("UserPromptSubmit", context_hook)
register_hook("PreToolUse", permission_hook)
register_hook("PreToolUse", log_hook)
register_hook("PostToolUse", large_output_hook)
register_hook("Stop", summary_hook)


def execute_tool(block, handlers:dict[str, callable]) -> str:

    blocked = trigger_hooks("PreToolUse", block)
    if (blocked):

        return (str(blocked))
    #
    handler = handlers.get(block.name)
    if (not handler):

        return (f"Unknown tool: {block.name}")
    #
    try:

        output = str(handler(**block.input))
    #
    except Exception as exc:

        output = f"Error: {type(exc).__name__}: {exc}"
    #
    trigger_hooks("PostToolUse", block, output)
    return (output)


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Agent loop with a dynamic tool pool
#

def agent_loop(messages:list):

    while (True):

        try:

            tools, handlers = assemble_tool_pool()
            response        = client.messages.create(
                model      = MODEL,
                system     = assemble_system_prompt(),
                messages   = messages,
                tools      = tools,
                max_tokens = 8000,
            )
        #
        except Exception as exc:

            messages.append({
                "role"    : "assistant",
                "content" : [{
                    "type" : "text",
                    "text" : f"[Error] {type(exc).__name__}: {exc}",
                }],
            })
            trigger_hooks("Stop", messages)
            return
        #

        messages.append({"role": "assistant", "content": response.content})
        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (not tool_calls):

            trigger_hooks("Stop", messages)
            return
        #

        results = []
        for block in tool_calls:

            print(f"> {block.name}")
            output = execute_tool(block, handlers)
            print(output[:300])
            results.append({
                "type"        : "tool_result",
                "tool_use_id" : block.id,
                "content"     : output,
            })
        #
        messages.append({"role": "user", "content": results})
    #
#


if (RUN_AGENT_INTERACTIVELY):

    print("s14: MCP tools")
    print("Enter a question, press Enter to send. Type q to quit.\n")
    history = []

    while (True):

        try:

            query = input("s14 >> ")
        #
        except (EOFError, KeyboardInterrupt):

            break
        #
        if (query.strip().lower() in {"q", "exit", ""}):

            break
        #
        trigger_hooks("UserPromptSubmit", query)
        history.append({"role": "user", "content": query})
        agent_loop(history)
        for block in history[-1].get("content", []):

            if (getattr(block, "type", None) == "text"):

                print(block.text)
            #
            elif (isinstance(block, dict)  and  block.get("type") == "text"):

                print(block.get("text", ""))
            #
        #
        print()
    #
#
